In [ ]:
"""
Residual Test — Is there nonlinear signal left after Linear Regression?
==========================================================================
Dissertation: Predicting Revenue Growth and Cost Reduction from Business AI Adoption
 
PURPOSE
-------
Diagnoses WHY XGBoost/Random Forest don't outperform the Linear Regression
baseline (RQ1 finding). Rather than assuming "noise ceiling", this directly
tests it:
 
  1. Take the Linear Regression model's predictions on the test set
  2. Compute residuals = y_actual - y_pred_LR
  3. Fit a FRESH XGBoost model to predict those residuals from the same
     features
  4. Check that model's R² on a held-out split of the residuals
 
INTERPRETATION
--------------
  R²(residuals) ~ 0   -> No nonlinear/interaction signal remains after the
                          linear fit. Confirms the "noise ceiling" framing
                          as a TESTED result, not just an interpretation.
  R²(residuals) >> 0   -> There IS nonlinear structure XGBoost's original
                          fit isn't capturing. Points to tuning/feature
                          engineering, not an unbeatable ceiling.
 
This reuses the EXACT same feature set, train/test split, and random seed
as shared_pipeline.py, and loads the ALREADY-TRAINED Linear Regression
pipeline from disk rather than retraining it, so this test is directly
comparable to your existing RQ1 results.
 
Run: python residual_test.py
Requires: shared_pipeline.py must already have been run (FULL run) for
          both targets, so pipeline_outputs/models/{target}_linear_regression.pkl
          exists for both.
"""
 
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
 
# ============================================================
# CONFIG -- must match shared_pipeline.py exactly, so the train/test
# split reproduces the SAME rows the saved LR model was evaluated on
# ============================================================
RAW_DATA_PATH = "ai_company_adoption.csv"   # <-- adjust to your actual filename
MODELS_DIR = "pipeline_outputs/models"
OUT_DIR = "pipeline_outputs/residual_test"
os.makedirs(OUT_DIR, exist_ok=True)
 
TARGET_1 = "revenue_growth_percent"
TARGET_2 = "cost_reduction_percent"
TARGET_1_LABEL = "Revenue Growth"
TARGET_2_LABEL = "Cost Reduction"
 
NUMERIC_FEATURES = [
    "ai_adoption_rate", "ai_budget_percentage", "ai_investment_per_employee",
    "ai_training_hours", "years_using_ai", "ai_projects_active",
    "task_automation_rate", "productivity_change_percent",
    "ai_risk_management_score", "regulatory_compliance_score",
    "annual_revenue_usd_millions", "num_employees",
]
CATEGORICAL_FEATURES = [
    "ai_adoption_stage", "ai_ethics_committee", "data_privacy_level",
    "industry", "company_size", "region",
]
 
RANDOM_SEED = 42       # <-- confirm this matches your actual shared_pipeline.py value
TEST_SIZE = 0.20
 
# XGBoost params for the residual model -- same primary-model config
# your grid search already found best for the ORIGINAL targets, reused
# here rather than re-running a full grid search on residuals (residuals
# are a diagnostic, not a new production model)
XGB_PARAMS = dict(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
 
# ============================================================
# LOAD DATA
# ============================================================
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw data: {df_raw.shape[0]:,} rows")
 
keep_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_1, TARGET_2]
missing = [c for c in keep_cols if c not in df_raw.columns]
assert not missing, f"Missing expected columns: {missing}"
df = df_raw[keep_cols].copy()
 
feature_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES
 
 
def run_residual_test(target_column, target_label):
    print("\n" + "=" * 80)
    print(f"RESIDUAL TEST — {target_label} ({target_column})")
    print("=" * 80)
 
    # ── 1. Reproduce the EXACT same train/test split used by shared_pipeline.py ──
    X = df[feature_cols]
    y = df[target_column]
    stratify_col = df["company_size"] if "company_size" in df.columns else None
 
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=stratify_col
    )
    print(f"Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
 
    # ── 2. Load the ALREADY-TRAINED Linear Regression pipeline (not retrained) ──
    lr_path = f"{MODELS_DIR}/{target_column}_linear_regression.pkl"
    if not os.path.exists(lr_path):
        raise FileNotFoundError(
            f"\nCould not find '{lr_path}'.\n"
            f"Run shared_pipeline.py (FULL run) for target_column='{target_column}' first."
        )
    with open(lr_path, "rb") as f:
        lr_pipeline = pickle.load(f)
 
    lr_pred_test = lr_pipeline.predict(X_test)
    lr_r2 = r2_score(y_test, lr_pred_test)
    print(f"Loaded existing Linear Regression model. Test R² = {lr_r2:.4f} "
          f"(should match your original RQ1 result)")
 
    # ── 3. Compute residuals on the TRAIN set (what the residual model learns from) ──
    # and TEST set (what we evaluate the residual model on) separately, using
    # the SAME train/test rows as above -- no data leakage between them.
    lr_pred_train = lr_pipeline.predict(X_train)
    residuals_train = y_train.values - lr_pred_train
    residuals_test = y_test.values - lr_pred_test
 
    print(f"Residuals (test set) — mean: {residuals_test.mean():.4f}, "
          f"std: {residuals_test.std():.4f}")
 
    # ── 4. Fit a FRESH XGBoost model to predict the residuals ──
    # Reuses the same preprocessing approach as shared_pipeline.py's tree
    # branch (one-hot encode categoricals, numerics passed through --
    # tree models are scale-invariant).
    X_train_encoded = pd.get_dummies(X_train, columns=CATEGORICAL_FEATURES, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test, columns=CATEGORICAL_FEATURES, drop_first=True)
    # Align columns in case a rare category appears in only one split
    X_train_encoded, X_test_encoded = X_train_encoded.align(
        X_test_encoded, join="left", axis=1, fill_value=0
    )
 
    residual_model = XGBRegressor(**XGB_PARAMS)
    residual_model.fit(X_train_encoded, residuals_train)
    residual_pred = residual_model.predict(X_test_encoded)
 
    residual_r2 = r2_score(residuals_test, residual_pred)
    residual_rmse = np.sqrt(mean_squared_error(residuals_test, residual_pred))
    residual_mae = mean_absolute_error(residuals_test, residual_pred)
 
    print(f"\nXGBoost fit to LR's residuals:")
    print(f"  R²   = {residual_r2:.4f}")
    print(f"  RMSE = {residual_rmse:.4f}")
    print(f"  MAE  = {residual_mae:.4f}")
 
    # ── 5. Interpretation ──
    if residual_r2 < 0.02:
        verdict = ("No meaningful nonlinear signal remains after the linear fit. "
                   "This CONFIRMS the noise-ceiling interpretation as a tested result, "
                   "not just an inference from similar model scores.")
    elif residual_r2 < 0.05:
        verdict = ("A small amount of residual structure exists, but it is marginal. "
                   "Weakly supports the noise-ceiling interpretation, with a minor caveat.")
    else:
        verdict = ("Meaningful nonlinear/interaction signal remains unexplained by the "
                   "linear model. This does NOT support a pure noise-ceiling interpretation "
                   "-- the original XGBoost model may be undertuned or missing structure.")
 
    print(f"\nVerdict: {verdict}")
 
    result = {
        "target": target_column,
        "target_label": target_label,
        "lr_test_r2": round(float(lr_r2), 4),
        "residual_r2": round(float(residual_r2), 4),
        "residual_rmse": round(float(residual_rmse), 4),
        "residual_mae": round(float(residual_mae), 4),
        "verdict": verdict,
    }
    return result
 
 
results = []
for target, label in [(TARGET_1, TARGET_1_LABEL), (TARGET_2, TARGET_2_LABEL)]:
    results.append(run_residual_test(target, label))
 
results_df = pd.DataFrame(results)
results_df.to_csv(f"{OUT_DIR}/residual_test_results.csv", index=False)
 
print("\n" + "=" * 80)
print("RESIDUAL TEST COMPLETE")
print("=" * 80)
print(results_df.to_string(index=False))
print(f"\nSaved: {OUT_DIR}/residual_test_results.csv")
 
print("\n--- Suggested dissertation text (Discussion, RQ1 noise-ceiling framing) ---")
for r in results:
    print(
        f"\n'A residual diagnostic test was conducted for {r['target_label'].lower()}, in which an "
        f"XGBoost model was fit to the residuals of the Linear Regression baseline. This model "
        f"achieved an R² of {r['residual_r2']:.3f} on held-out residuals, indicating "
        f"{'negligible' if r['residual_r2'] < 0.02 else 'minimal' if r['residual_r2'] < 0.05 else 'non-trivial'} "
        f"remaining nonlinear or interaction signal beyond what the linear model already captured. "
        f"This supports the interpretation that the observed performance ceiling reflects a genuine "
        f"limit in the dataset's learnable signal, rather than a shortfall in model selection or "
        f"tuning.'"
    )